# Temperature Forecasting with LSTM (PyTorch)

This notebook demonstrates **time-series forecasting** using **RNN/LSTM** with PyTorch.
- **Dataset**: Synthetic temperature data with trend and seasonal patterns
- **Model**: LSTM-based neural network
- **Task**: Predict future temperatures based on historical data
- **Compatible**: Works in local Jupyter and Google Colab

## Section 1: Colab Setup and Dependencies

Install required packages. Run this cell first if using Google Colab.

In [ ]:
# Install required packages (for Colab compatibility)
# Uncomment if running in Google Colab
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# !pip install pandas numpy matplotlib scikit-learn

import sys
print("Python version:", sys.version)
print("Installation successful! Ready to proceed.")
print("=" * 60)

## Section 2: Import Libraries and Device Detection

Import all necessary libraries and configure PyTorch device (CPU/GPU)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device configuration (CPU or GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print("=" * 60)

## Section 3: Load or Create Simple Temperature Dataset

Generate synthetic temperature time-series data with trend + seasonal patterns + noise

In [ ]:
# Generate synthetic temperature data
# Formula: temperature = baseline + trend + seasonal + noise

n_days = 500  # Number of days
baseline = 20  # Baseline temperature (°C)

# Create time index
time = np.arange(n_days)

# Add trend (slight increase over time)
trend = 0.02 * time

# Add seasonal pattern (sine wave with 365-day period)
seasonal = 10 * np.sin(2 * np.pi * time / 365)

# Add random noise
noise = np.random.normal(0, 1, n_days)

# Combine all components
temperature = baseline + trend + seasonal + noise

# Create DataFrame
df = pd.DataFrame({
    'Day': time,
    'Temperature': temperature
})

print("=" * 60)
print("DATASET CREATION")
print("=" * 60)
print(f"Dataset shape: {df.shape}")
print(f"Date range: Day 0 to Day {n_days-1}")
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Statistics:")
print(df['Temperature'].describe())
print("=" * 60)

## Section 4: Visualize Time Series and Basic Statistics

In [ ]:
# Visualize the time series
plt.figure(figsize=(14, 5))
plt.plot(df['Day'], df['Temperature'], linewidth=1, color='blue')
plt.xlabel('Day', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.title('Synthetic Temperature Time Series (500 days)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
print("\n" + "=" * 60)
print("TIME SERIES STATISTICS")
print("=" * 60)
print(f"Mean temperature: {df['Temperature'].mean():.2f}°C")
print(f"Std deviation: {df['Temperature'].std():.2f}°C")
print(f"Min temperature: {df['Temperature'].min():.2f}°C")
print(f"Max temperature: {df['Temperature'].max():.2f}°C")
print(f"Temperature range: {df['Temperature'].max() - df['Temperature'].min():.2f}°C")
print("\nSample data points:")
sample_indices = [0, 100, 200, 300, 400, 499]
for idx in sample_indices:
    print(f"  Day {df.loc[idx, 'Day']}: {df.loc[idx, 'Temperature']:.2f}°C")
print("=" * 60)

## Section 5: Prepare Supervised Sequences (Sliding Window)

Convert time series to supervised learning format using sliding window approach.

**Formula**: $y_t = f(x_{t-k:t-1})$ - Predict temperature at time $t$ based on previous $k$ observations

In [ ]:
# Normalize the data
scaler = MinMaxScaler(feature_range=(0, 1))
data_normalized = scaler.fit_transform(df['Temperature'].values.reshape(-1, 1)).flatten()

print("=" * 60)
print("DATA NORMALIZATION")
print("=" * 60)
print(f"Original data range: [{df['Temperature'].min():.2f}, {df['Temperature'].max():.2f}]")
print(f"Normalized data range: [{data_normalized.min():.2f}, {data_normalized.max():.2f}]")

# Create sliding window sequences
def create_sequences(data, window_size):
    """
    Create input-output pairs using sliding window.
    X: sequences of length window_size
    y: next value after each sequence
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i + window_size])
        y.append(data[i + window_size])
    return np.array(X), np.array(y)

window_size = 20  # Use 20 previous days to predict next day
X, y = create_sequences(data_normalized, window_size)

print(f"\nSequence creation:")
print(f"Window size (lookback): {window_size} days")
print(f"Total sequences created: {len(X)}")
print(f"X shape: {X.shape} (samples, sequence_length)")
print(f"y shape: {y.shape} (samples,)")

print(f"\nExample:")
print(f"  Input sequence (first 20 days): {X[0]}")
print(f"  Target (day 21): {y[0]:.4f}")
print("=" * 60)

## Section 6: PyTorch Dataset and DataLoader

Create a custom PyTorch Dataset class and prepare train/val/test splits

In [ ]:
# Create custom PyTorch Dataset
class TemperatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split data into train, val, test (60%, 20%, 20%)
train_size = int(len(X) * 0.6)
val_size = int(len(X) * 0.2)

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

# Create datasets
train_dataset = TemperatureDataset(X_train, y_train)
val_dataset = TemperatureDataset(X_val, y_val)
test_dataset = TemperatureDataset(X_test, y_test)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("=" * 60)
print("DATA SPLITTING AND DATALOADER")
print("=" * 60)
print(f"Total samples: {len(X)}")
print(f"Train samples: {len(X_train)} ({100*len(X_train)/len(X):.1f}%)")
print(f"Val samples: {len(X_val)} ({100*len(X_val)/len(X):.1f}%)")
print(f"Test samples: {len(X_test)} ({100*len(X_test)/len(X):.1f}%)")
print(f"Batch size: {batch_size}")

# Display sample batch
print(f"\nSample batch from train loader:")
sample_seq, sample_target = next(iter(train_loader))
print(f"  Batch shape - Sequences: {sample_seq.shape}, Targets: {sample_target.shape}")
print(f"  First sequence in batch (normalized): {sample_seq[0]}")
print(f"  First target in batch (normalized): {sample_target[0].item():.4f}")
print("=" * 60)

## Section 7: Define LSTM Forecasting Model

Build a LSTM-based neural network architecture.

**LSTM equation**: $$h_t = \text{LSTM}(x_t, h_{t-1})$$

Where $h_t$ is the hidden state at time $t$, carrying information from all previous inputs.

In [ ]:
# Define LSTM model
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1):
        super(LSTMForecaster, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(input_size=input_size, 
                            hidden_size=hidden_size, 
                            num_layers=num_layers, 
                            batch_first=True)
        
        # Fully connected layer for output
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x shape: (batch_size, sequence_length, input_size)
        
        # LSTM forward pass
        lstm_out, (h_n, c_n) = self.lstm(x)
        # lstm_out shape: (batch_size, sequence_length, hidden_size)
        
        # Use last timestep's output
        last_output = lstm_out[:, -1, :]
        # last_output shape: (batch_size, hidden_size)
        
        # Fully connected layer
        output = self.fc(last_output)
        # output shape: (batch_size, output_size)
        
        return output

# Model parameters
input_size = 1  # Input feature dimension (just temperature)
hidden_size = 64  # Number of LSTM hidden units
num_layers = 2  # Number of LSTM layers

# Initialize model
model = LSTMForecaster(input_size=input_size, 
                       hidden_size=hidden_size, 
                       num_layers=num_layers)
model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("LSTM MODEL ARCHITECTURE")
print("=" * 60)
print(model)
print(f"\nModel Configuration:")
print(f"  Input size: {input_size}")
print(f"  Hidden size: {hidden_size}")
print(f"  Number of LSTM layers: {num_layers}")
print(f"  Output size: 1")
print(f"\nModel Parameters:")
print(f"  Total parameters: {total_params}")
print(f"  Trainable parameters: {trainable_params}")
print(f"  Device: {device}")

# Test forward pass with dummy input
dummy_input = torch.randn(batch_size, window_size, input_size).to(device)
with torch.no_grad():
    dummy_output = model(dummy_input)
print(f"\nForward pass test:")
print(f"  Input shape: {dummy_input.shape}")
print(f"  Output shape: {dummy_output.shape}")
print("=" * 60)

## Section 8: Training Loop with Per-Epoch Prints

Train the LSTM model on the training dataset

In [ ]:
# Training setup
criterion = nn.MSELoss()  # Mean Squared Error Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, 
                                                  patience=5, verbose=True)

num_epochs = 50
train_losses = []
val_losses = []

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        # Reshape for LSTM: (batch_size, sequence_length) -> (batch_size, sequence_length, 1)
        X_batch = X_batch.unsqueeze(-1)
        
        # Forward pass
        y_pred = model(X_batch)
        
        # Reshape target for loss calculation
        y_batch = y_batch.unsqueeze(-1)
        
        # Calculate loss
        loss = criterion(y_pred, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            X_batch = X_batch.unsqueeze(-1)
            
            y_pred = model(X_batch)
            y_batch = y_batch.unsqueeze(-1)
            
            loss = criterion(y_pred, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Print progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        
        # Show sample prediction
        if epoch == 0 or (epoch + 1) % 20 == 0:
            with torch.no_grad():
                sample_batch, sample_target = next(iter(val_loader))
                sample_batch = sample_batch.to(device).unsqueeze(-1)
                sample_pred = model(sample_batch)
                print(f"  Sample - Predicted: {sample_pred[0].item():.6f}, Actual: {sample_target[0].item():.6f}")

print("=" * 60)
print(f"Training completed!")
print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")
print("=" * 60)

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 9: Evaluate on Test Set and Plot Results

Evaluate model performance on the test set with metrics and visualization

In [ ]:
# Evaluate on test set
model.eval()
all_predictions = []
all_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device).unsqueeze(-1)
        y_batch = y_batch.to(device)
        
        y_pred = model(X_batch)
        
        all_predictions.extend(y_pred.cpu().numpy().flatten())
        all_actuals.extend(y_batch.cpu().numpy().flatten())

all_predictions = np.array(all_predictions)
all_actuals = np.array(all_actuals)

# Calculate metrics
mse = mean_squared_error(all_actuals, all_predictions)
mae = mean_absolute_error(all_actuals, all_predictions)
rmse = np.sqrt(mse)

print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print(f"Mean Squared Error (MSE):  {mse:.6f}")
print(f"Mean Absolute Error (MAE): {mae:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")

print(f"\nSample Predictions vs Actual:")
for i in range(min(5, len(all_predictions))):
    print(f"  Sample {i+1} - Predicted: {all_predictions[i]:.6f}, Actual: {all_actuals[i]:.6f}")
print("=" * 60)

# Plot predictions vs actual
plt.figure(figsize=(14, 5))
plt.plot(all_actuals, label='Actual', linewidth=2, alpha=0.7)
plt.plot(all_predictions, label='Predicted', linewidth=2, alpha=0.7)
plt.xlabel('Test Sample', fontsize=12)
plt.ylabel('Temperature (normalized)', fontsize=12)
plt.title('Predicted vs Actual Temperature (Test Set)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Section 10: Save and Load Model Checkpoint

Save the trained model for future use and demonstrate loading it

In [ ]:
# Save model checkpoint
checkpoint_path = 'temperature_forecasting_model.pth'

checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_losses[-1],
    'val_loss': val_losses[-1],
    'model_config': {
        'input_size': input_size,
        'hidden_size': hidden_size,
        'num_layers': num_layers
    }
}

torch.save(checkpoint, checkpoint_path)
print("=" * 60)
print("MODEL CHECKPOINT")
print("=" * 60)
print(f"Model saved to: {checkpoint_path}")

# Load model from checkpoint
print(f"\nLoading model from checkpoint...")

loaded_checkpoint = torch.load(checkpoint_path)

# Create a new model instance
loaded_model = LSTMForecaster(input_size=loaded_checkpoint['model_config']['input_size'],
                              hidden_size=loaded_checkpoint['model_config']['hidden_size'],
                              num_layers=loaded_checkpoint['model_config']['num_layers'])
loaded_model.load_state_dict(loaded_checkpoint['model_state_dict'])
loaded_model.to(device)
loaded_model.eval()

print(f"Model loaded successfully!")
print(f"  Epoch: {loaded_checkpoint['epoch']}")
print(f"  Final training loss: {loaded_checkpoint['train_loss']:.6f}")
print(f"  Final validation loss: {loaded_checkpoint['val_loss']:.6f}")

# Test loaded model
print(f"\nTesting loaded model:")
with torch.no_grad():
    test_sample = next(iter(test_loader))
    test_X, test_y = test_sample
    test_X = test_X.to(device).unsqueeze(-1)
    
    # Original model prediction
    original_pred = model(test_X)
    
    # Loaded model prediction
    loaded_pred = loaded_model(test_X)
    
    print(f"  Original model output:  {original_pred[0].item():.6f}")
    print(f"  Loaded model output:    {loaded_pred[0].item():.6f}")
    print(f"  Match: {torch.allclose(original_pred, loaded_pred, atol=1e-5)}")
print("=" * 60)

## Section 11: Predict Future Temperatures (Multi-step Forecasting)

Use the trained model to forecast future temperatures using autoregressive prediction

In [ ]:
# Multi-step forecasting
def forecast_future(model, last_sequence, steps=30):
    """
    Generate predictions for multiple steps ahead (autoregressive)
    
    Args:
        model: trained LSTM model
        last_sequence: initial sequence (normalized values)
        steps: number of future steps to predict
    
    Returns:
        predictions: array of predicted values (normalized)
    """
    model.eval()
    predictions = []
    current_seq = last_sequence.copy()
    
    with torch.no_grad():
        for _ in range(steps):
            # Prepare input
            input_tensor = torch.FloatTensor(current_seq).unsqueeze(0).unsqueeze(-1).to(device)
            
            # Predict next step
            pred = model(input_tensor).cpu().numpy()[0, 0]
            predictions.append(pred)
            
            # Update sequence: remove first value, append prediction
            current_seq = np.append(current_seq[1:], pred)
    
    return np.array(predictions)

# Use last sequence from training data for forecasting
last_sequence = X[-1]  # Last sequence from dataset

# Forecast next 30 days
forecast_steps = 30
future_predictions = forecast_future(model, last_sequence, steps=forecast_steps)

# Convert normalized predictions back to original scale
future_predictions_original = scaler.inverse_transform(future_predictions.reshape(-1, 1)).flatten()
last_actual_temp = df['Temperature'].iloc[-1]

print("=" * 60)
print("MULTI-STEP FORECASTING")
print("=" * 60)
print(f"Last known temperature: {last_actual_temp:.2f}°C (normalized: {data_normalized[-1]:.4f})")
print(f"\nForecast for next {forecast_steps} days (original scale):")
for i, pred in enumerate(future_predictions_original[:10]):
    print(f"  Day {i+1}: {pred:.2f}°C")
print(f"  ...")
print(f"  Day {forecast_steps}: {future_predictions_original[-1]:.2f}°C")

# Plot historical + forecasted data
plt.figure(figsize=(14, 6))

# Historical data
history_to_plot = 200  # Plot last 200 days
historical_data = df['Temperature'].iloc[-history_to_plot:].values
historical_days = df['Day'].iloc[-history_to_plot:].values

plt.plot(historical_days, historical_data, label='Historical', linewidth=2, color='blue')

# Forecasted data
forecast_days = np.arange(df['Day'].iloc[-1] + 1, df['Day'].iloc[-1] + 1 + forecast_steps)
plt.plot(forecast_days, future_predictions_original, label='Forecast', linewidth=2, 
         color='red', linestyle='--', marker='o', markersize=4)

plt.xlabel('Day', fontsize=12)
plt.ylabel('Temperature (°C)', fontsize=12)
plt.title('Temperature Forecast: Historical Data + Future Predictions', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("=" * 60)

## Section 12: Optional - Run on GPU in Google Colab

Instructions for enabling GPU and verifying CUDA availability

In [ ]:
# Instructions for Google Colab GPU setup:
# 1. Click "Runtime" in the menu
# 2. Click "Change runtime type"
# 3. Select "GPU" from the "Hardware accelerator" dropdown
# 4. Click "Save"
# Then run the cells below to verify

print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

# Check CUDA availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")

if torch.cuda.is_available():
    # Display GPU info
    print(f"\nGPU Information:")
    print(f"  Device name: {torch.cuda.get_device_name(0)}")
    print(f"  Device count: {torch.cuda.device_count()}")
    print(f"  Current device: {torch.cuda.current_device()}")
    
    # Try nvidia-smi if in Colab
    import os
    if os.path.exists('/opt/bin/nvidia-smi'):
        print("\nnvidia-smi output:")
        os.system('nvidia-smi')
else:
    print("\nGPU not available. Using CPU instead.")
    print("To enable GPU in Colab:")
    print("  1. Go to Runtime menu")
    print("  2. Select 'Change runtime type'")
    print("  3. Choose 'GPU' as hardware accelerator")

# Verify model is on correct device
print(f"\nModel device: {next(model.parameters()).device}")
print("=" * 60)

print("\n" + "=" * 60)
print("NOTEBOOK COMPLETE!")
print("=" * 60)
print("\nYou have successfully trained an LSTM model for temperature forecasting!")
print("\nKey concepts learned:")
print("  ✓ Time-series data generation and preprocessing")
print("  ✓ Sliding window approach for supervised learning")
print("  ✓ LSTM architecture and implementation")
print("  ✓ Training, validation, and testing")
print("  ✓ Model evaluation with metrics")
print("  ✓ Multi-step autoregressive forecasting")
print("  ✓ Model persistence (save/load)")
print("  ✓ PyTorch basics and GPU support")
print("=" * 60)